# Ingest generic files into `crawler.documents_b0`

Every supported file in a folder (pdf, docx, pptx, xlsx, csv, html, txt, ...) becomes one row: filename, text and inferred metadata (title, author, publication date, extension, size, modified time). For books split into chapters use `books_ingest.ipynb`.

In [ ]:
import sys
sys.path[:0] = ["../book_tools/book-parser/src", "../book_tools/book-ingest/src"]  # or pip install the packages

INBOX = "/home/user/Documents/inbox"  # folder with the files
SPARK_CONNECT_URL = "sc://localhost:15002"
TABLE = "crawler.documents_b0"

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote(SPARK_CONNECT_URL).getOrCreate()

## Preview one file (no Spark needed)

In [ ]:
from pathlib import Path
from book_ingest import FolderScanner
from book_parser import DocumentParser

first = FolderScanner(INBOX).pending()[0]
doc = DocumentParser().parse(first)
print(doc.filename, doc.title, doc.author, doc.publication_date, doc.size_bytes)
print(doc.text[:500])

## Ingest the folder

Processed files move to `processed/`, unreadable ones to `failed/`; files already in the table (same hash) are skipped.

In [ ]:
import logging
from book_ingest import DocumentsRepository, FileArchiver, IngestJob

logging.basicConfig(level=logging.INFO)
result = IngestJob(
    FolderScanner(INBOX),
    DocumentParser(),
    DocumentsRepository(spark, TABLE),
    FileArchiver(INBOX),
).run()
result

In [ ]:
spark.table(TABLE).select(
    "filename", "extension", "author", "publication_date", "char_count"
).show(30, truncate=60)